In [1]:
from PIL import Image, ImageOps, ImageFont
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets
from ipywidgets import interactive, HBox, VBox, interact
from modules.ansi_colorizer import AnsiColorizer, reset_code, set_char_fg_rgb_color_code, set_char_bg_rgb_color_code
from modules.img_processing import preprocess_img, DITHER_MODES
from modules.palette_generator import get_asciis
from modules.img2chars_mono_converter import Img2MonoCharsConverter, pick_closest

In [2]:
SYMBOLS = get_asciis()

In [3]:
FONT = ImageFont.truetype("../fonts/CascadiaMono.ttf", 11)
IMG_PATH = "../imgs/rad_grad_with_inv.bmp"

In [4]:
orig_img = Image.open(IMG_PATH).convert("RGB")

print(orig_img.size)

(512, 512)


In [5]:
def i_preprocess_img(
        scale_factor=0.2,
        contrast=1.0,
        brightness=1.0,
        sharpness=1.0,
        enhance_edges=0.0,
        quantize_colors=128,
        eq=0.0,
        dither=DITHER_MODES.NONE,
        grayscale=True):
    proc_img = preprocess_img(
        img=orig_img,
        scale_factor=scale_factor,
        contrast=contrast,
        brightness=brightness,
        eq=eq,
        quantize_colors=quantize_colors,
        dither=dither,
        sharpness=sharpness,
        enhance_edges=enhance_edges,
        grayscale=grayscale)
    
    plt.imshow(proc_img, cmap='gray', vmin=0, vmax=255, interpolation='none')
    plt.figure(figsize=(6, 3))
    plt.xticks([x for x in range(0, len(proc_img.histogram()), 50)])
    plt.bar([x for x in range(0, len(proc_img.histogram()))],
            proc_img.histogram())
    plt.grid()
    plt.show()
    return proc_img

interactive_preprocess = interactive(i_preprocess_img,
                                     scale_factor=(0.01, 1, 0.01),
                                     contrast=(0, 2, 0.01),
                                     brightness=(0, 2, 0.01),
                                     sharpness=(0, 2, 0.01),
                                     eq=(0, 1, 0.01),
                                     enhance_edges=(0, 1, 0.01),
                                     quantize_colors=(1, 256, 1),
                                     dither=DITHER_MODES,
                                     grayscale=True)

controls = VBox(interactive_preprocess.children[:-1])
display(HBox((controls, interactive_preprocess.children[-1])))

In [6]:
def i_convert_img(
        gen_map_btm_bins=8,
        gen_map_top_bins=8,
        detail_map_h=6,
        detail_map_w=3,
        dither=DITHER_MODES.NONE,
        colored_fg=False,
        colored_bg=False,
        fg_brightness=1.0,
        bg_brightness=1.0,
        use_ansi_256_colors=False):
    proc_img = interactive_preprocess.result
    
    colorize_settings = None
    if colored_fg or colored_bg:
        colorize_settings = AnsiColorizer(
            colored_fg=colored_fg,
            colored_bg=colored_bg,
            fg_brightness_scale=fg_brightness,
            bg_brightness_scale=bg_brightness,
            use_ansi_256_colors=use_ansi_256_colors
        )

    gen_map = (gen_map_btm_bins,)
    if gen_map_top_bins > 0:
        gen_map = (gen_map_btm_bins, gen_map_top_bins)

    detail_map_shape = None
    if detail_map_h and detail_map_w:
        detail_map_shape = (detail_map_h, detail_map_w)

    converter = Img2MonoCharsConverter(
        symbols=SYMBOLS,
        font=FONT,
        general_mapping_palette_shape=gen_map,
        detailed_mapping_win_shape=detail_map_shape,
        pick_strategy=pick_closest,
        dither=dither,
        ansi_colorizer=colorize_settings
    )

    char_arr = converter.convert(proc_img)

    for line in char_arr:
        for sym in line:
            if not colored_bg:
                print(set_char_bg_rgb_color_code(0,0,0), sep='', end='')
            if not colored_fg:
                print(set_char_fg_rgb_color_code(255,255,255), sep='', end='')
            print(sym, sep='', end='')
        print('\n', end='')
    print(reset_code())
    
interactive_convert = interactive(i_convert_img,
                                  gen_map_btm_bins=(1, 24),
                                  gen_map_top_bins=(0, 24),
                                  detail_map_h=(0, 16),
                                  detail_map_w=(0, 8),
                                  dither=DITHER_MODES,
                                  colored_fg=False,
                                  colored_bg=False,
                                  fg_brightness=(0.0, 1.5),
                                  bg_brightness=(0.0, 1.5),
                                  use_ansi_256_colors=False)

display(interactive_convert)


interactive(children=(IntSlider(value=8, description='gen_map_btm_bins', max=24, min=1), IntSlider(value=8, de…